# 10 — Identity (Identity Platform + IAM)

**AWS-side equivalent:** Cognito + AVP
**GCP service:** Identity Platform (the customer-facing CIAM product) + `firebase_admin` SDK + IAM principal sets

AWS-side notebook: `notebooks/aws/10-agentcore-identity.ipynb`

_As of 2026-05: Identity Platform remains the GCP brand for B2C/CIAM; the Firebase Auth API surface is the shared admin SDK (`firebase_admin.auth` + `firebase_admin.tenant_mgt`). For a workforce/SSO use case, Workforce Identity Federation is the better fit — this notebook covers the CIAM path since OrgIQ's end users live outside the corporate directory._


## Steps

1. Initialise `firebase_admin` against the project
2. Create (or look up) an Identity Platform tenant for OrgIQ
3. Seed three demo users, one per role, with `custom_claims = {role: ...}`
4. Verify a JWT and translate its `role` claim into the GCP principal set defined in notebook 09
5. Sketch the Cloud Function pattern that maps claims → IAM (Identity-Aware Proxy / Workforce Federation glue)


## Step 1: Load config


In [ ]:
import json
from pathlib import Path

config_path = Path('config.json')
config = json.loads(config_path.read_text())
gcp = config.setdefault('gcp', {})
PROJECT_ID = gcp.get('project_id')
REGION = gcp.get('region', 'us-central1')
assert PROJECT_ID, 'Run 01-setup-credentials.ipynb first.'

# Pulled from notebook 09; falls back to the same shape if 09 hasn't run yet.
ROLE_BINDINGS = gcp.get('role_bindings', {
    'member':     {'gcp_role': 'roles/aiplatform.user'},
    'org-leader': {'gcp_role': 'roles/aiplatform.user'},
    'hr-admin':   {'gcp_role': 'roles/iam.securityReviewer'},
})
print(f'project={PROJECT_ID} region={REGION}  roles={list(ROLE_BINDINGS)}')


## Step 2: Initialise `firebase_admin`

`firebase_admin.initialize_app()` works with ADC — no service-account key needed in the notebook environment. The Identity Platform tenant APIs live in `firebase_admin.tenant_mgt`; per-tenant user CRUD is reached via `tenant_mgt.auth_for_tenant(tenant_id)`.


In [ ]:
try:
    import firebase_admin
    from firebase_admin import auth as fb_auth, tenant_mgt
except ImportError:
    print('⚠ firebase-admin not installed — `pip install firebase-admin>=6.5.0`')
    raise

if not firebase_admin._apps:
    firebase_admin.initialize_app(options={'projectId': PROJECT_ID})
print(f'✓ firebase_admin initialised against project {PROJECT_ID}')


## Step 3: Create (or look up) the OrgIQ tenant

Identity Platform supports multi-tenancy: a tenant is a namespace of users + IdP configs scoped under the same project. We create `orgiq-demo` once and reuse it on reruns.


In [ ]:
TENANT_DISPLAY_NAME = 'orgiq-demo'
tenant_id = gcp.get('tenant_id')

tenant = None
if tenant_id:
    try:
        tenant = tenant_mgt.get_tenant(tenant_id)
        print(f'✓ Reusing tenant {tenant.tenant_id} ({tenant.display_name})')
    except tenant_mgt.TenantNotFoundError:
        tenant_id = None

if tenant is None:
    # List + match by display name to avoid duplicates if config.json was wiped.
    for t in tenant_mgt.list_tenants().iterate_all():
        if t.display_name == TENANT_DISPLAY_NAME:
            tenant = t
            break

if tenant is None:
    tenant = tenant_mgt.create_tenant(
        display_name=TENANT_DISPLAY_NAME,
        allow_password_sign_up=True,
        enable_email_link_sign_in=False,
    )
    print(f'✓ Created tenant {tenant.tenant_id}')

tenant_id = tenant.tenant_id
tenant_auth = tenant_mgt.auth_for_tenant(tenant_id)
print(f'tenant_id={tenant_id}')


## Step 4: Seed three demo users, one per role

Custom claims live on the user record and are mirrored into every ID token the user receives. The middleware reads `decoded_token['role']` and looks up the GCP principal in the mapping from notebook 09. Idempotent: re-running this cell updates claims on the existing user instead of failing on `EmailAlreadyExistsError`.


In [ ]:
DEMO_USERS = [
    {'email': 'alice@orgiq.example',  'role': 'member',     'display_name': 'Alice Member'},
    {'email': 'bob@orgiq.example',    'role': 'org-leader', 'display_name': 'Bob Leader'},
    {'email': 'carol@orgiq.example',  'role': 'hr-admin',   'display_name': 'Carol HR'},
]

seeded = []
for spec in DEMO_USERS:
    try:
        user = tenant_auth.get_user_by_email(spec['email'])
    except fb_auth.UserNotFoundError:
        user = tenant_auth.create_user(
            email=spec['email'],
            email_verified=True,
            display_name=spec['display_name'],
            password='ChangeMe-Demo-2026!',  # demo only — rotate before real use.
        )
    tenant_auth.set_custom_user_claims(user.uid, {'role': spec['role'], 'tenant': tenant_id})
    seeded.append({'uid': user.uid, 'email': spec['email'], 'role': spec['role']})
    print(f'  {spec["email"]:30s} uid={user.uid[:12]}… role={spec["role"]}')

print(f'\n✓ Seeded {len(seeded)} users with custom claims')


## Step 5: Mint a custom token and decode it

We can't run a true OAuth sign-in flow from a notebook (that needs a browser). Instead we use the Admin SDK to **mint a custom token** for one of the demo users, then exchange it via the Identity Platform REST endpoint to produce a real ID token whose `role` claim we can verify. The REST exchange requires a Web API key — fetch one from the [Identity Platform console](https://console.cloud.google.com/customer-identity) and drop it in `config.json` under `gcp.web_api_key`.


In [ ]:
import urllib.request, urllib.error

WEB_API_KEY = gcp.get('web_api_key')
demo_uid = seeded[0]['uid']  # alice — member

custom_token = tenant_auth.create_custom_token(demo_uid).decode()
print(f'✓ Minted custom token for {seeded[0]["email"]} (len={len(custom_token)})')

if WEB_API_KEY:
    url = f'https://identitytoolkit.googleapis.com/v1/accounts:signInWithCustomToken?key={WEB_API_KEY}'
    body = json.dumps({
        'token': custom_token,
        'returnSecureToken': True,
        'tenantId': tenant_id,
    }).encode()
    req = urllib.request.Request(url, data=body, headers={'Content-Type': 'application/json'})
    try:
        with urllib.request.urlopen(req, timeout=10) as resp:
            payload = json.loads(resp.read())
        id_token = payload['idToken']
        decoded = fb_auth.verify_id_token(id_token)
        print(f'✓ Verified ID token; role claim = {decoded.get("role")!r}, tenant = {decoded.get("firebase", {}).get("tenant")!r}')
    except urllib.error.HTTPError as e:
        print(f'⚠ REST exchange failed: {e.code} {e.read().decode()[:200]}')
else:
    print('⚠ gcp.web_api_key not set in config.json — skipping live JWT exchange.')
    print('  The custom token would still be verifiable client-side once the user signs in.')


## Step 6: Claim → GCP principal mapping

The middleware is the policy enforcement point. Pseudo-code for the Cloud Function / Cloud Run handler that consumes the JWT:

```python
from firebase_admin import auth

def authorize(request):
    decoded = auth.verify_id_token(request.headers['Authorization'].split()[1])
    role = decoded.get('role', 'member')
    gcp_role = ROLE_BINDINGS[role]['gcp_role']
    # The principal exposed to downstream IAM:
    principal = f'principal://iam.googleapis.com/projects/{PROJECT_ID}/locations/global/workforcePools/orgiq/subject/{decoded["uid"]}'
    return principal, gcp_role
```

For workforce SSO you would wire Identity Platform as an OIDC provider to Workforce Identity Federation and let IAM consume the federated principal directly (no Cloud Function needed). For CIAM (this notebook) the middleware does the translation explicitly because Identity Platform users are not first-class IAM principals on their own.


In [ ]:
# Demonstrate the mapping in-process for each seeded user.
for u in seeded:
    binding = ROLE_BINDINGS[u['role']]
    principal = (
        f'principal://iam.googleapis.com/projects/{PROJECT_ID}'
        f'/locations/global/workforcePools/orgiq/subject/{u["uid"]}'
    )
    print(f'{u["email"]:30s} role={u["role"]:11s} → {binding["gcp_role"]}')
    print(f'    principal: {principal}')


## Step 7: Persist tenant + user info to `config.json`


In [ ]:
gcp['tenant_id'] = tenant_id
gcp['tenant_display_name'] = TENANT_DISPLAY_NAME
gcp['demo_users'] = seeded
tracked = gcp.setdefault('tracked_resources', [])
if {'kind': 'identityplatform.tenant', 'id': tenant_id} not in tracked:
    tracked.append({'kind': 'identityplatform.tenant', 'id': tenant_id})
config_path.write_text(json.dumps(config, indent=2))
print(f'✓ config.json updated (tenant_id={tenant_id}, {len(seeded)} users tracked)')


## Done

Next: `11-code-execution.ipynb` (give Gemini a Python sandbox that respects the role claim above).
